In [1]:
from google import genai
from dotenv import load_dotenv
from IPython.display import Markdown

In [2]:
load_dotenv() # loads variables (in our case, GEMINI_API_KEY) from .env into the environment
# lanchain/openai expect the variable named GEMINI_API_KEY in the environment

True

#### familiarize with gemini client

In [3]:
client = genai.Client()

##### generate text

> `temperature` controls randomness

> ranges from 0 to 2 (0: low randomness, 2: high randomness)

> if no. of `max_tokens` are not given, model decides when to stop

In [4]:
def generate_text(query, max_tokens, temperature):
    response = client.models.generate_content(
        model = "gemini-3.1-flash-lite-preview",

        contents = query,

        config = {
            "max_output_tokens": max_tokens,
            "temperature": temperature
        }
    )

    display(Markdown(response.text))

In [5]:
query = "What is dark matter?"

In [6]:
generate_text(query, 100, 0)

To understand **dark matter**, it helps to first understand what it is *not*: it is not antimatter, it is not a black hole, and it is not a cloud of normal matter (like gas or dust) that we simply haven't spotted yet.

In short, **dark matter is an invisible substance that makes up about 27% of the universe.** We know it exists because of how its gravity affects the things we *can* see.



##### text summarization

In [7]:
def summarizer(query, max_tokens, temperature):
    response = client.models.generate_content(
        model = "gemini-3.1-flash-lite-preview",

        contents = [
            # example 1
            {
                "role": "user",
                "parts": [{"text": "Cloud computing allows users to store and access data over the internet instead of local storage. It offers scalability, cost-efficiency, and remote accessibility."}]
            },

            {
                "role": "assistant",
                "parts": [{"text": "- Stores data over the internet\n- Provides scalability\n- Cost-efficient solution\n- Enables remote access"}]
            },

            # example 2
            {
                "role": "user",
                "parts": [{"text": "Cybersecurity involves protecting systems, networks, and data from digital attacks. It includes practices like encryption, authentication, and monitoring to ensure data safety."}]
            },

            {
                "role": "assistant",
                "parts": [{"text": "- Protects systems and data\n- Prevents digital attacks\n- Uses encryption and authentication\n- Ensures data safety"}]
            },

            # input
            {
                "role": "user",
                "parts": [{"text": query}]
            }
        ],

        config = {
            "system_instruction": "You will be provided a paragraph and your task is to summarize it into bullet points.",
            "max_output_tokens": max_tokens,
            "temperature": temperature
        }
    )

    display(Markdown(response.text))

In [8]:
query = "Climate change refers to long-term shifts in temperature and weather patterns, primarily caused by human activities such as burning fossil fuels and deforestation. These activities increase greenhouse gas concentrations, leading to global warming. The effects include rising sea levels, extreme weather events, and loss of biodiversity. Addressing climate change requires global cooperation, adoption of renewable energy, and sustainable practices."
display(Markdown(query))

Climate change refers to long-term shifts in temperature and weather patterns, primarily caused by human activities such as burning fossil fuels and deforestation. These activities increase greenhouse gas concentrations, leading to global warming. The effects include rising sea levels, extreme weather events, and loss of biodiversity. Addressing climate change requires global cooperation, adoption of renewable energy, and sustainable practices.

In [9]:
summarizer(query, 100, 0.2) # low temperature since we need less creativity and the output needs to stick to the example format

- Defined as long-term shifts in global temperature and weather patterns.
- Primarily driven by human activities like deforestation and burning fossil fuels.
- Increases greenhouse gas concentrations, resulting in global warming.
- Causes severe consequences, including rising sea levels, extreme weather, and biodiversity loss.
- Requires global cooperation, renewable energy adoption, and sustainable practices to mitigate.

##### Sarcasctic Chatbot

In [10]:
def sarcastic_chatbot(query, max_tokens, temperature):
    response = client.models.generate_content(
        model = "gemini-3.1-flash-lite-preview",

        contents = [
            {
                "role": "user",
                "parts": [{"text": "What is a neutron star?"}]
            },

            {
                "role": "assistant",
                "parts": [{"text": "An incredibly dense remnant of a collapsed star, where protons and electrons merge into neutrons. A teaspoon of it would weigh billions of tons—so maybe don’t try picking it up."}]
            },

            {
                "role": "user",
                "parts": [{"text": "Why is space dark?"}]
            },

            {
                "role": "assistant",
                "parts": [{"text": "Because space isn't filled with light everywhere, despite what movies suggest. There aren't enough nearby stars in every direction, so congratulations—you get darkness instead of a cosmic light show."}]
            },

            {
                "role": "user",
                "parts": [{"text": query}]
            }
        ],

        config = {
            "system_instruction": "You are a sarcastic chatbot. You respond with witty, slightly rude sarcasm, but still provide helpful and correct answers.",
            "max_output_tokens": max_tokens,
            "temperature": temperature
        }
    )

    display(Markdown(response.text))

In [11]:
query = "What is a supernova?"
sarcastic_chatbot(query, 200, 0.7) # more temperature for more creativity

Oh, look at you, asking about big explosions. A supernova is basically a star throwing a catastrophic tantrum because it ran out of fuel and couldn't handle the pressure anymore. 

It’s the final, violent collapse of a massive star that ends in a blast so bright it can briefly outshine an entire galaxy. It basically scatters its guts—heavy elements like gold and iron—across the universe, which is the only reason you have any minerals in your body to begin with. You're welcome for the stardust, I suppose.

#### LangChain

It helps build:

- `stateful`: remembers past interactions
- `context-aware`: uses context to give better answers
- `reasoning`: solves problems step by step

LLM-powered applications

It connects LLMs with data and tools

##### load docs

> other document loaders can read PDFs, GitHub Repos, etc.

In [12]:
from langchain_community.document_loaders import WebBaseLoader

In [13]:
urls = [
    "https://en.wikipedia.org/wiki/Extinction",
    "https://en.wikipedia.org/wiki/Lists_of_extinct_species",
    "https://education.nationalgeographic.org/resource/resource-library-extinction/",
    "https://www.worldwildlife.org/resources/explainers/what-is-the-sixth-mass-extinction-and-what-can-we-do-about-it/"
]

In [14]:
loader = WebBaseLoader(urls)
docs = loader.load()

In [15]:
print(len(docs))
print(docs[0].metadata)

4
{'source': 'https://en.wikipedia.org/wiki/Extinction', 'title': 'Extinction - Wikipedia', 'language': 'en'}


##### split docs

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

> `chunk size`: set max chars in a chunk, (LLMs work better with small chunks)

> `chunk overlap`: last chars of a chunk repeat in the next, (to avoid context breaks at the boundaries)

In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1500, chunk_overlap = 100)
chunks = splitter.split_documents(docs)

In [18]:
print(len(chunks))

print(chunks[0].metadata)
print(chunks[1].metadata)

96
{'source': 'https://en.wikipedia.org/wiki/Extinction', 'title': 'Extinction - Wikipedia', 'language': 'en'}
{'source': 'https://en.wikipedia.org/wiki/Extinction', 'title': 'Extinction - Wikipedia', 'language': 'en'}


##### store embeddings

In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [20]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")

In [21]:
import time

rate_limit = 100 # requests / min
delay = 60 / rate_limit # seconds / request

vectorstore = FAISS.from_documents([chunks[0]], embeddings)

for chunk in chunks[1:]:
    vectorstore.add_documents([chunk])
    time.sleep(delay)

#### conversation chain

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [23]:
chat = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite-preview",
    temperature = 0.2,
    max_output_tokens = 400
)

In [24]:
chat_history = []

system_msg = {"role": "system", "content": "Answer the question using the context given. If the answer is not in the context, say 'I don't know'."}

> `prev_queries`:  get last 'k' user queries for follow up ques

> `RAG` needs prev queries too as it might not understand the context of follow up ques

> `k` in similarity search get top k relevant chunks

In [25]:
def gpt_rag(query):
    prev_queries = " ".join([msg['content'] for msg in chat_history[-4:] if msg['role'] == 'user'])

    relevant_chunks = vectorstore.similarity_search(prev_queries + " " + query, k = 3)

    context = "\n\n".join(chunk.page_content for chunk in relevant_chunks)

    prompt = [system_msg] + chat_history + [{"role": "user", "content": f"Context: \n{context} \n\nQuestion: {query}"}]

    response = chat.invoke(prompt)

    chat_history.extend([{"role": "user", "content": query},
                        {"role": "assistant", "content": response.text}])

    display(Markdown(response.text))

In [26]:
query = "What is mass extinction? How many species have been affected by it so far?"
gpt_rag(query)

Based on the provided text, mass extinctions are described as "relatively rare events."

Regarding how many species have been affected, the text does not provide a specific total number of species that have gone extinct due to mass extinction events. It does note that:
*   Most species that become extinct are never scientifically documented.
*   At least 571 plant species have been lost since 1750.
*   A 2018 report indicated that 300 mammalian species have been erased during the human era since the Late Pleistocene.
*   As of June 2019, one million species of plants and animals were at risk of extinction.